# GWAS Logistic Regression Pipeline
Parallelized plink2 GLM across ancestries and chromosomes, followed by results concatenation and gwaslab plots.

In [ ]:
import os
import subprocess
import glob
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
import gwaslab as gl

## Configuration

In [ ]:
WD = "/path/to/working/directory" 

ANCESTRIES   = ["CAS", "MDE", "EUR"]
CHROMOSOMES  = list(range(1, 23))                      
PHENOTYPE    = "DISEASE"                               
LINEAR_METRIC = None                                   

# Per ancestry covariate file
# COVAR = f"{WD}/CAS/CAS_GWAS/covariate.txt"

MAX_WORKERS = 4   
PLINK_THREADS = 8

## Parallelize jobs per ancestry and chromosome

In [ ]:
def logisticRegression(ancestry, chromosome, phenotype, wd, plink_threads):
    """
    Run plink2 QC filter + GLM logistic regression for one ancestry/chromosome
    Return (ancestry, chromosome, output_prefix) on successful completion
    """
    input_pfile = f"{wd}/{ancestry}/{ancestry}_imputed_qc_genotools/chr{chromosome}_qcd"
    output      = f"{wd}/{ancestry}/{ancestry}_GWAS/chr{chromosome}"
    covar      = f"{wd}/{ancestry}/{ancestry}_GWAS/covariate.txt"
    out_dir     = os.path.dirname(output)
    os.makedirs(out_dir, exist_ok=True)

    # Step 1: QC filter 
    qc_pfile = f"{input_pfile}.qc"
    filtervars = [
        "plink2",
        "--pfile",        input_pfile,
        "--snps-only",    "just-acgt",
        "--maf",          "0.05",
        "--mac",          "10",
        "--hwe",          "1e-5",
        "--pheno",        covar,
        "--pheno-col-nums", "4",
        "--threads",      str(plink_threads),
        "--make-pgen",
        "--out",          qc_pfile,
    ]
    subprocess.run(filtervars, check=True,
                   capture_output=True, text=True)

    # Step 2: GLM logistic regression 
    glmLogGWAS = [
        "plink2",
        "--pfile",   qc_pfile,
        "--glm",
            "hide-covar",
            "firth-fallback",
            "cols=+beta,+a1freqcc,+a1countcc,+a1count,+totallele,+totallelecc,+gcountcc,+ci",
        "--ci",        "0.95",
        "--adjust",
        "--pheno",     covar,
        "--pheno-name", phenotype,
        "--require-pheno", phenotype,
        "--covar",     covar,
        "--covar-name", "SEX,AGE,PC1,PC2,PC3,PC4,PC5",
        "--covar-variance-standardize",
        "--threads",   str(plink_threads),
        "--out",       output,
    ]
    subprocess.run(glmLogGWAS, check=True,
                   capture_output=True, text=True)

    return ancestry, chromosome, output

## Parallelized run

In [ ]:
jobs = [
    (anc, chrom)
    for anc   in ANCESTRIES
    for chrom in CHROMOSOMES
]

results = []   # (ancestry, chromosome, output_prefix)
failed  = []   # (ancestry, chromosome, error_message)

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {
        pool.submit(
            logisticRegression,
            anc, chrom, PHENOTYPE, WD, PLINK_THREADS
        ): (anc, chrom)
        for anc, chrom in jobs
    }

    for future in as_completed(futures):
        anc, chrom = futures[future]
        try:
            results.append(future.result())
            print(f"[COMPLETE] {anc}  chr{chrom}")
        except subprocess.CalledProcessError as e:
            failed.append((anc, chrom, str(e)))
            print(f"[ERROR] {anc}  chr{chrom}: {e}")

print(f"\nCompleted {len(results)}/{len(jobs)} jobs.")
if failed:
    print("Failed jobs:", failed)

## Concatenate GLM output files per ancestry

In [ ]:
# plink2 logistic names the result file  <out>.DISEASE.glm.logistic.hybrid
GLM_SUFFIX = f".{PHENOTYPE}.glm.logistic.hybrid"

combined = {}   # ancestry → DataFrame

for ancestry in ANCESTRIES:
    gwas_dir = f"{WD}/{ancestry}/{ancestry}_GWAS"
    pattern  = f"{gwas_dir}/chr*{GLM_SUFFIX}"
    files    = sorted(glob.glob(pattern))

    if not files:
        print(f"[{ancestry}] No GLM output files found, skipping.")
        continue

    dfs = []
    for f in files:
        df = pd.read_csv(f, sep="\t", comment="#", low_memory=False)
        dfs.append(df)

    merged = pd.concat(dfs, ignore_index=True)
    out_path = f"{WD}/{ancestry}/{ancestry}_GWAS/{ancestry}_all_chr{GLM_SUFFIX}"
    merged.to_csv(out_path, sep="\t", index=False)
    combined[ancestry] = merged
    print(f"[{ancestry}] {len(merged):,} variants written to {out_path}")

## Make Gwaslab plots (Manhattan + QQ) per ancestry

In [ ]:
for ancestry, df in combined.items():
    print(f"\n {ancestry} ")

    # Rename plink2 columns to gwaslab-compatible colnames
    df = df.rename(columns={
        "#CHROM": "CHR",
        "POS":    "BP",
        "ID":     "SNP",
        "REF":    "NEA",
        "ALT":    "EA",
        "BETA":   "BETA",
        "SE":     "SE",
        "P":      "P",
    })

    # Keep only ADD  rows
    if "TEST" in df.columns:
        df = df[df["TEST"] == "ADD"].copy()

    mysumstats = gl.Sumstats(
        df,
        snpid  = "SNP",
        chrom  = "CHR",
        pos    = "BP",
        ea     = "EA",
        nea    = "NEA",
        beta   = "BETA",
        se     = "SE",
        p      = "P",
        build  = "38",    
    )

    plot_prefix = f"{WD}/{ancestry}/{ancestry}_GWAS/{ancestry}"

    # Manhattan plot
    mysumstats.plot_mqq(
        mode       = "m",
        title      = f"{ancestry} GWAS: {PHENOTYPE}",
        save       = f"{plot_prefix}_manhattan.png",
        saveargs   = {"dpi": 300, "bbox_inches": "tight"},
    )

    # QQ plot
    mysumstats.plot_mqq(
        mode       = "qq",
        title      = f"{ancestry} QQ: {PHENOTYPE}",
        save       = f"{plot_prefix}_qq.png",
        saveargs   = {"dpi": 300, "bbox_inches": "tight"},
    )

    print(f"  Plots saved: {plot_prefix}_manhattan.png / _qq.png")